# Day 9 · 회전 표현: 쿼터니언


| Part | 주제 |
|------|------|
| Part 1 | 쿼터니언의 정의, 시각화, 보간(SLERP) |

```
pip install ipywidgets ipympl plotly
pip install "nbformat>=4.2.0"
```

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import interact, FloatSlider
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# 3D 플롯을 마우스로 회전·확대할 수 있도록 interactive 백엔드 설정
# (VS Code Interactive 창이나 Jupyter에서 실행 시 적용됩니다)
%matplotlib widget

In [3]:
# 공통으로 사용할 회전 행렬 함수들
def rot_x(theta):
    """X축 회전 행렬 (roll)"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1, 0, 0],
                     [0, c, -s],
                     [0, s, c]])


def rot_y(theta):
    """Y축 회전 행렬 (pitch)"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s],
                     [0, 1, 0],
                     [-s, 0, c]])


def rot_z(theta):
    """Z축 회전 행렬 (yaw)"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0],
                     [s, c, 0],
                     [0, 0, 1]])

In [4]:
def euler_to_rotation_matrix(roll, pitch, yaw, order='ZYX'):
    """
    오일러 각도 → 회전 행렬
    로봇 공학에서 가장 흔한 순서: Z-Y-X (yaw → pitch → roll)
    """
    R_x = rot_x(roll)
    R_y = rot_y(pitch)
    R_z = rot_z(yaw)
    axis_map = {'X': R_x, 'Y': R_y, 'Z': R_z}

    # order 문자열을 거꾸로 읽어 행렬을 곱합니다.
    # 예: 'ZYX' -> R_z @ R_y @ R_x
    R = np.eye(3)
    for axis in reversed(order):
        R = axis_map[axis] @ R
    return R

In [5]:
def draw_axes_3d(ax, R, origin=(0, 0, 0), scale=1.0,
                 colors=('r', 'g', 'b'), labels=None,
                 alpha=1.0, linewidth=2):
    """
    주어진 회전 행렬 R에 따라 3D 좌표축을 그립니다.
    R: 3x3 회전 행렬
    origin: 좌표축의 원점
    colors: (X축 색, Y축 색, Z축 색)
    """
    axes_vec = np.eye(3) * scale
    origin = np.array(origin)
    for i, (axis, color) in enumerate(zip(axes_vec, colors)):
        rotated = R @ axis
        ax.plot([origin[0], origin[0] + rotated[0]],
                [origin[1], origin[1] + rotated[1]],
                [origin[2], origin[2] + rotated[2]],
                color=color, linewidth=linewidth, alpha=alpha)
        if labels:
            ax.text(origin[0] + rotated[0] * 1.15,
                    origin[1] + rotated[1] * 1.15,
                    origin[2] + rotated[2] * 1.15,
                    labels[i], color=color, fontsize=10, fontweight='bold')

In [6]:
def set_3d_axis_equal(ax, limit=1.5):
    """3D 축의 스케일을 동일하게 맞춥니다."""
    ax.set_xlim([-limit, limit])
    ax.set_ylim([-limit, limit])
    ax.set_zlim([-limit, limit])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

In [7]:
# ───────────────────────────────────────────────
# Plotly 3D 시각화 공통 헬퍼 함수
# ───────────────────────────────────────────────
def _add_plotly_axes(fig, R=None, scale=1.0, row=None, col=None,
                     names=None, colors=None, line_width=3):
    """Plotly Figure에 3D 좌표축을 추가합니다."""
    axes = np.eye(3) * scale
    if names is None:
        names = ["X", "Y", "Z"]
    if colors is None:
        colors = ["#888888", "#888888", "#888888"]

    kwargs = {}
    if row is not None:
        kwargs['row'] = row
    if col is not None:
        kwargs['col'] = col

    for axis, name, color in zip(axes, names, colors):
        vec = R @ axis if R is not None else axis
        fig.add_trace(go.Scatter3d(
            x=[0, vec[0]], y=[0, vec[1]], z=[0, vec[2]],
            mode="lines+text",
            line=dict(color=color, width=line_width),
            text=["", name],
            textposition="top center",
            showlegend=False
        ), **kwargs)


def _plotly_scene(limit=1.5, eye=None):
    """Plotly 3D scene 공통 레이아웃을 반환합니다."""
    scene = dict(
        xaxis=dict(range=[-limit, limit], title="X", gridcolor="lightgray"),
        yaxis=dict(range=[-limit, limit], title="Y", gridcolor="lightgray"),
        zaxis=dict(range=[-limit, limit], title="Z", gridcolor="lightgray"),
        aspectmode="cube",
    )
    if eye is not None:
        scene["camera"] = dict(eye=eye)
    return scene


def plot_rotation_trajectory(rotations, title, n_arrows=6):
    """
    여러 회전 행렬의 궤적을 3D로 시각화합니다.
    rotations: (n_steps, 3, 3) 회전 행렬 리스트
    """
    fig = go.Figure()
    
    # 원래 좌표계 (World)
    _add_plotly_axes(fig, scale=1.2,
                     names=["X", "Y", "Z"],
                     colors=["#444444", "#444444", "#444444"],
                     line_width=3)
    
    n_steps = len(rotations)
    
    # 각 축(X', Y', Z')의 궤적 (끝점들을 선으로 연결)
    axis_colors = ["#e74c3c", "#2ecc71", "#3498db"]
    axis_names = ["X'", "Y'", "Z'"]
    
    for axis_idx in range(3):
        points = np.array([R[:, axis_idx] for R in rotations])
        fig.add_trace(go.Scatter3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2],
            mode='lines',
            line=dict(color=axis_colors[axis_idx], width=5),
            name=f"{axis_names[axis_idx]} trajectory",
            showlegend=True
        ))
        
        # 궤적 위의 주요 지점 마커
        step = max(1, n_steps // 10)
        fig.add_trace(go.Scatter3d(
            x=points[::step, 0],
            y=points[::step, 1],
            z=points[::step, 2],
            mode='markers',
            marker=dict(color=axis_colors[axis_idx], size=4, opacity=0.5),
            showlegend=False
        ))
    
    # 주요 단계의 좌표계 표시 (t=0, 중간, t=1)
    display_indices = [0, n_steps//4, n_steps//2, 3*n_steps//4, n_steps-1]
    for idx in display_indices:
        R = rotations[idx]
        t = idx / (n_steps - 1)
        alpha = 0.4 + 0.6 * t
        lw = 2 + 4 * t
        _add_plotly_axes(fig, R=R, scale=1.0,
                         names=[f"X'", f"Y'", f"Z'"],
                         colors=[
                             f"rgba(231,76,60,{alpha})",
                             f"rgba(46,204,113,{alpha})",
                             f"rgba(52,152,219,{alpha})"
                         ],
                         line_width=lw)
    
    fig.update_layout(
        title=dict(text=title, x=0.5),
        scene=_plotly_scene(1.5),
        width=700,
        height=650,
        margin=dict(l=0, r=0, b=0, t=40),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    return fig

---
## Part 3: 쿼터니언(Quaternion)

짐벌락과 보간 문제를 해결하는 **4차원 수 체계**입니다.

### 3.1 쿼터니언의 정의와 직관

쿼터니언은 **하나의 실수 + 세 개의 허수**로 구성됩니다:

$$ q = w + xi + yj + zk = [w, x, y, z] $$

| 구성요소 | 의미 |
|----------|------|
| **w** | 실수부 — 회전량의 크기 (코사인 절반) |
| **(x, y, z)** | 허수부 — 회전 축의 방향 (사인 절반) |

**단위 쿼터니언**은 회전을 표현하기 위해 크기를 1로 제한합니다:

$$ |q| = \sqrt{w^2 + x^2 + y^2 + z^2} = 1 $$

이 조건 하에서 쿼터니언과 회전의 관계는 다음과 같습니다:

| 기호 | 의미 | 수식 |
|------|------|------|
| $\theta$ | 회전각 | $\theta = 2 \arccos(w)$ |
| $\hat{n}$ | 회전축 (단위 벡터) | $\hat{n} = (x, y, z) / \sin(\theta/2)$ |

즉, 단위 쿼터니언은 **"어떤 축을 중심으로 얼마나 회전할지"**를 한 번에 담고 있습니다:

$$ q = \left[ \cos\frac{\theta}{2},\ \sin\frac{\theta}{2} \cdot \hat{n} \right] $$

> **왜 4차원이 필요한가?**
>
> 3D 공간의 회전은 3개의 각도로 표현할 수 있지만(오일러 각도),
> 짐벌락이나 보간 문제를 해결하려면 **4차원 공간에서의 단위 구면 위에 점으로 표현**하는 것이 자연스럽습니다.
> 마치 지구의 위도·경도(2D)로는 극점에서 특이점이 생기는 것처럼,
> 3D 회전을 3개의 각도로 표현하면 특정 자세에서 특이점(짐벌락)이 생깁니다.

> **핵심 직관**: "회전을 **한 번에 하나의 축 + 각도**로 표현"
>
> - 오일러: X 돌리고 Y 돌리고 Z 돌림 (3단계)
> - 쿼터니언: "특정 축을 중심으로 한 번에 원하는 만큼 돌림" (1단계)
>
> 이게 왜 중요한가? 회전 축이 하나라서 **짐벌락이 생길 물리적 공간이 없습니다**.

In [8]:
# ───────────────────────────────────────────────
# 쿼터니언 기본 연산 함수들
# ───────────────────────────────────────────────

def euler_to_quaternion(roll, pitch, yaw):
    """
    오일러 각도 (ZYX 순서) → 단위 쿼터니언
    입력: roll, pitch, yaw (라디안)
    출력: [w, x, y, z]
    """
    cy = np.cos(yaw * 0.5)
    sy = np.sin(yaw * 0.5)
    cp = np.cos(pitch * 0.5)
    sp = np.sin(pitch * 0.5)
    cr = np.cos(roll * 0.5)
    sr = np.sin(roll * 0.5)

    w = cr * cp * cy + sr * sp * sy
    x = sr * cp * cy - cr * sp * sy
    y = cr * sp * cy + sr * cp * sy
    z = cr * cp * sy - sr * sp * cy

    q = np.array([w, x, y, z])
    return q / np.linalg.norm(q)   # 단위 쿼터니언으로 정규화


def quaternion_to_rotation_matrix(q):
    """
    단위 쿼터니언 → 3x3 회전 행렬
    """
    w, x, y, z = q
    return np.array([
        [1 - 2*(y**2 + z**2), 2*(x*y - z*w),     2*(x*z + y*w)],
        [2*(x*y + z*w),     1 - 2*(x**2 + z**2), 2*(y*z - x*w)],
        [2*(x*z - y*w),     2*(y*z + x*w),     1 - 2*(x**2 + y**2)]
    ])


def quaternion_multiply(q1, q2):
    """두 쿼터니언의 곱 (합성 회전)"""
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    w = w1*w2 - x1*x2 - y1*y2 - z1*z2
    x = w1*x2 + x1*w2 + y1*z2 - z1*y2
    y = w1*y2 - x1*z2 + y1*w2 + z1*x2
    z = w1*z2 + x1*y2 - y1*x2 + z1*w2
    return np.array([w, x, y, z])


def quaternion_conjugate(q):
    """쿼터니언의 켤레 (역회전)"""
    return np.array([q[0], -q[1], -q[2], -q[3]])


def rotate_vector_by_quaternion(v, q):
    """벡터 v를 쿼터니언 q로 회전"""
    qv = np.array([0, v[0], v[1], v[2]])
    q_conj = quaternion_conjugate(q)
    result = quaternion_multiply(quaternion_multiply(q, qv), q_conj)
    return result[1:]


def slerp(q1, q2, t):
    """
    구면 선형 보간 (Spherical Linear Interpolation)
    두 쿼터니언 q1, q2 사이를 부드럽게 보간합니다.
    t: 0.0 ~ 1.0
    """
    q1 = q1 / np.linalg.norm(q1)
    q2 = q2 / np.linalg.norm(q2)

    dot = np.dot(q1, q2)

    # 최단 경로 보장 (dot < 0 이면 반대 방향으로)
    if dot < 0.0:
        q2 = -q2
        dot = -dot

    dot = np.clip(dot, -1.0, 1.0)
    theta_0 = np.arccos(dot)
    sin_theta_0 = np.sin(theta_0)

    # 두 쿼터니언이 거의 같으면 선형 보간으로 대체 (수치 안정성)
    if sin_theta_0 < 1e-6:
        return (1.0 - t) * q1 + t * q2

    theta = theta_0 * t
    sin_theta = np.sin(theta)

    s1 = np.cos(theta) - dot * sin_theta / sin_theta_0
    s2 = sin_theta / sin_theta_0

    q_interp = (s1 * q1) + (s2 * q2)
    return q_interp / np.linalg.norm(q_interp)

In [9]:
# 쿼터니언 예시: 오일러 각도와 비교
roll = np.deg2rad(30)
pitch = np.deg2rad(20)
yaw = np.deg2rad(45)

q = euler_to_quaternion(roll, pitch, yaw)
R_from_q = quaternion_to_rotation_matrix(q)
R_from_euler = euler_to_rotation_matrix(roll, pitch, yaw)

print("=== 쿼터니언 ===")
print(f"q = [{q[0]:.4f}, {q[1]:.4f}, {q[2]:.4f}, {q[3]:.4f}]")
print(f"|q| = {np.linalg.norm(q):.6f}  (단위 쿼터니언이어야 1)")
print("\n=== 쿼터니언 → 회전 행렬 ===")
print(np.round(R_from_q, 4))
print("\n=== 오일러 → 회전 행렬 ===")
print(np.round(R_from_euler, 4))
print(f"\n두 행렬이 동일? {np.allclose(R_from_q, R_from_euler)}")

=== 쿼터니언 ===
q = [0.8960, 0.1713, 0.2525, 0.3225]
|q| = 1.000000  (단위 쿼터니언이어야 1)

=== 쿼터니언 → 회전 행렬 ===
[[ 0.6645 -0.4915  0.563 ]
 [ 0.6645  0.7333 -0.1441]
 [-0.342   0.4698  0.8138]]

=== 오일러 → 회전 행렬 ===
[[ 0.6645 -0.4915  0.563 ]
 [ 0.6645  0.7333 -0.1441]
 [-0.342   0.4698  0.8138]]

두 행렬이 동일? True


### 3.2 쿼터니언의 직관적 이해 — 회전축과 회전각

단위 쿼터니언 $q = [w, x, y, z]$는 다음과 같이 해석할 수 있습니다:

- **회전축 (단위 벡터)**: $\hat{n} = (x, y, z) / \sqrt{x^2+y^2+z^2}$
- **회전각**: $\theta = 2 \arccos(w)$

즉, "어떤 축을 중심으로 얼마나 돌릴지"를 **한 번에** 표현합니다.

In [10]:
# 쿼터니언을 회전축 + 회전각으로 해석
q = euler_to_quaternion(np.deg2rad(30), np.deg2rad(20), np.deg2rad(45))
w, x, y, z = q

# 회전축 추출
axis = np.array([x, y, z])
axis_norm = np.linalg.norm(axis)
if axis_norm > 1e-6:
    axis_unit = axis / axis_norm
else:
    axis_unit = np.array([0, 0, 1])

# 회전각 (라디안)
theta = 2 * np.arccos(np.clip(w, -1.0, 1.0))

print(f"쿼터니언: [{w:.4f}, {x:.4f}, {y:.4f}, {z:.4f}]")
print(f"회전축:   {np.round(axis_unit, 4)} (단위 벡터)")
print(f"회전각:   {np.rad2deg(theta):.2f}°")
print("\n→ '이 축을 중심으로 이 각도만큼 한 번에 회전'한다고 생각하면 됩니다.")

쿼터니언: [0.8960, 0.1713, 0.2525, 0.3225]
회전축:   [0.3858 0.5687 0.7264] (단위 벡터)
회전각:   52.72°

→ '이 축을 중심으로 이 각도만큼 한 번에 회전'한다고 생각하면 됩니다.


### 3.3 쿼터니언으로 3D 회전 시각화

쿼터니언을 회전 행렬로 변환하여 시각화합니다.
결과는 오일러 각도와 동일하지만, 표현 방식이 다릅니다.

In [11]:
# 쿼터니언 기반 3D 회전 시각화
roll = np.deg2rad(30)
pitch = np.deg2rad(20)
yaw = np.deg2rad(45)

q = euler_to_quaternion(roll, pitch, yaw)
R_q = quaternion_to_rotation_matrix(q)

fig = go.Figure()
_add_plotly_axes(fig, scale=1.0,
                 names=["X", "Y", "Z"],
                 colors=["#444444", "#444444", "#444444"])
_add_plotly_axes(fig, R=R_q, scale=1.0,
                 names=["X'", "Y'", "Z'"],
                 colors=["#e74c3c", "#2ecc71", "#3498db"],
                 line_width=5)

fig.update_layout(
    title=dict(
        text="Quaternion Rotation<br><sub>Same as Euler (30°, 20°, 45°) but represented differently</sub>",
        x=0.5
    ),
    scene=_plotly_scene(1.5),
    width=700,
    height=650,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

### 3.4 SLERP 보간 — 쿼터니언의 진가

쿼터니언은 **구면 선형 보간(SLERP)**으로
두 자세 사이를 **항상 최단 경로, 일정한 속도**로 이동시킵니다.

In [12]:
# SLERP 보간 — 오일러 각도 직접 보간과의 비교
# 동일한 회전: (0°, 0°, 0°) → (150°, 60°, 120°)

q_start = euler_to_quaternion(0, 0, 0)
q_end = euler_to_quaternion(np.deg2rad(150), np.deg2rad(60), np.deg2rad(120))

n_steps = 50
print("=== Quaternion SLERP: (0°, 0°, 0°) → (150°, 60°, 120°) ===\n")
rotations_slerp = []
for i in range(n_steps):
    t = i / (n_steps - 1)
    q_interp = slerp(q_start, q_end, t)
    R_interp = quaternion_to_rotation_matrix(q_interp)
    rotations_slerp.append(R_interp)
    if i % 5 == 0:
        print(f"t = {t:.1f}")
        with np.printoptions(precision=3, suppress=True):
            print(R_interp)
        print()

print("쿼터니언 SLERP는 회전 공간에서 항상 최단 경로를 따릅니다.\n")

=== Quaternion SLERP: (0°, 0°, 0°) → (150°, 60°, 120°) ===

t = 0.0
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

t = 0.1
[[ 0.982  0.019  0.19 ]
 [-0.004  0.997 -0.075]
 [-0.191  0.073  0.979]]

t = 0.2
[[ 0.927  0.051  0.372]
 [ 0.006  0.989 -0.149]
 [-0.375  0.141  0.916]]

t = 0.3
[[ 0.838  0.095  0.537]
 [ 0.03   0.975 -0.219]
 [-0.544  0.2    0.815]]

t = 0.4
[[ 0.72   0.149  0.678]
 [ 0.067  0.957 -0.282]
 [-0.691  0.249  0.679]]

t = 0.5
[[ 0.576  0.212  0.789]
 [ 0.116  0.935 -0.335]
 [-0.809  0.285  0.514]]

t = 0.6
[[ 0.414  0.279  0.866]
 [ 0.174  0.91  -0.377]
 [-0.894  0.306  0.328]]

t = 0.7
[[ 0.239  0.35   0.906]
 [ 0.239  0.883 -0.404]
 [-0.941  0.313  0.128]]

t = 0.8
[[ 0.06   0.419  0.906]
 [ 0.308  0.856 -0.416]
 [-0.95   0.304 -0.078]]

t = 0.9
[[-0.116  0.485  0.867]
 [ 0.378  0.829 -0.413]
 [-0.918  0.28  -0.28 ]]

쿼터니언 SLERP는 회전 공간에서 항상 최단 경로를 따릅니다.



In [13]:
# SLERP 보간의 3D 궤적 시각화
fig = plot_rotation_trajectory(
    rotations_slerp,
    title="Quaternion SLERP Interpolation<br><sub>(0°, 0°, 0°) → (150°, 60°, 120°)</sub>"
)
fig.show()